# E4: RAG com FAISS - Busca Semântica

**MBA IA Generativa PCDF - IBMEC**  
**Encontro 4:** RAG (Retrieval-Augmented Generation)

---

## Objetivos

1. **Manter** as 8 tools do E3
2. **Adicionar** busca semântica com RAG
3. **Integrar** dados estruturados + não-estruturados
4. **Responder** perguntas conceituais

**Tempo estimado:** 5 horas

**Progressão:** E4 AGREGA ao E3, não substitui!

---

## PARTE 1: RECAP E3

### O que construímos no E3:

**8 Tools Funcionais:**
1. `contar_armas_marca` - Conta por marca
2. `contar_armas_calibre` - Conta por calibre
3. `contar_armas_tipo` - Conta por tipo
4. `contar_armas_combinado` - Marca + tipo
5. `ranking_marcas` - TOP 5 marcas
6. `ranking_calibres` - TOP 5 calibres
7. `estatisticas_gerais` - Resumo completo
8. `distribuicao_marca_por_tipo` - Distribuição

**Recursos:**
- Decorators (@tool, @lru_cache)
- Roteador inteligente
- Validação de segurança
- Cache de dados

**Limitação do E3:**
- ❌ Só responde perguntas sobre **dados estruturados** (CSV)
- ❌ Não responde perguntas **conceituais** ("O que é calibre?")

**Solução do E4:**
- ✅ RAG para perguntas conceituais
- ✅ Mantém tools E3 para dados estruturados

---

## PASSO 1: Instalação e Imports

In [1]:
# Instalar dependências (executar apenas uma vez)
# !pip install pandas langchain-core scikit-learn faiss-cpu

In [2]:
# Imports necessários
import pandas as pd
from functools import lru_cache
from langchain_core.tools import tool
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import os

print("✅ Imports realizados com sucesso!")

✅ Imports realizados com sucesso!


---

## PASSO 2: Carregar Dados E3 (Estruturados)

In [3]:
@lru_cache(maxsize=1)
def carregar_csv():
    """
    Carrega dados SINARM do CSV.
    Cache garante que carrega apenas UMA VEZ.
    """
    # Tentar diferentes caminhos
    caminhos_possiveis = [
        "../01_DADOS/DADOS_SINARM/OCORRENCIAS/OCORRENCIAS_2026.csv"
    ]
    
    caminho = None
    for c in caminhos_possiveis:
        if os.path.exists(c):
            caminho = c
            break
    
    if not caminho:
        raise FileNotFoundError(f"Arquivo não encontrado em nenhum dos caminhos: {caminhos_possiveis}")
    
    # Tentar diferentes configurações
    configs = [
        {'encoding': 'utf-8', 'sep': ','},
        {'encoding': 'utf-8', 'sep': ';'},
        {'encoding': 'latin-1', 'sep': ','},
        {'encoding': 'latin-1', 'sep': ';'},  # ⭐ ESTE É O CORRETO!
        {'encoding': 'iso-8859-1', 'sep': ';'},
        {'encoding': 'cp1252', 'sep': ';'},
    ]
    
    for config in configs:
        try:
            df = pd.read_csv(caminho, **config)
            
            # Validar se carregou corretamente (deve ter múltiplas colunas)
            if len(df.columns) > 1:
                print(f"[CACHE] Carregando CSV com encoding={config['encoding']}, sep='{config['sep']}'")
                print(f"[OK] {len(df)} registros, {len(df.columns)} colunas carregadas!")
                print(f"[COLUNAS] {list(df.columns)[:5]}...")
                return df
        except (UnicodeDecodeError, pd.errors.ParserError):
            continue
    
    raise Exception(f"Não foi possível ler o arquivo com nenhuma configuração testada")

# Testar carregamento
df = carregar_csv()
print(f"\n📊 Primeiros registros:")
df.head()

[CACHE] Carregando CSV com encoding=latin-1, sep=';'
[OK] 74758 registros, 10 colunas carregadas!
[COLUNAS] ['ANO_OCORRENCIA', 'MES_OCORRENCIA', 'UF', 'MUNICIPIO', 'ESPECIE_ARMA']...

📊 Primeiros registros:


,ANO_OCORRENCIA,MES_OCORRENCIA,UF,MUNICIPIO,ESPECIE_ARMA,MARCA_ARMA,CALIBRE_ARMA,TIPO_OCORRENCIA,MAIS_1000_MIL_HAB,TOTAL
0,2026,1,AC,ACRELÂNDIA,Espingarda ...,BOITO (E.R. AMANTINO & CIA) ...,.32 ...,...,N,1
1,2026,1,AC,ACRELÂNDIA,Espingarda ...,BOITO (E.R. AMANTINO & CIA) ...,20 ...,...,N,1
2,2026,1,AC,ACRELÂNDIA,Espingarda ...,BOITO (E.R. AMANTINO & CIA) ...,28 ...,...,N,1
3,2026,1,AC,ACRELÂNDIA,Espingarda ...,BOITO (E.R. AMANTINO & CIA) ...,36 ...,...,N,1
4,2026,1,AC,ACRELÂNDIA,Espingarda ...,CBC (COMPANHIA BRASILEIRA DE CARTUCHOS) ...,.32 ...,...,N,4


---

## PASSO 3: Tools E3 (Mantidas)

Vamos manter as 8 tools do E3 funcionando.

In [4]:
# TOOL 1: Contar armas por marca
@tool
def contar_armas_marca(marca: str) -> str:
    """
    Conta quantas armas de uma marca específica existem no banco.
    
    Args:
        marca: Nome da marca (ex: TAURUS, GLOCK, BERETTA)
    
    Returns:
        String com o resultado da contagem
    """
    df = carregar_csv()
    
    # Busca parcial (aceita variações)
    resultado = df[df['MARCA_ARMA'].str.contains(marca, case=False, na=False)]
    total = len(resultado)
    
    if total == 0:
        return f"Não encontrei armas da marca '{marca}'"
    
    marca_real = resultado['MARCA_ARMA'].iloc[0]
    return f"Encontrei {total} armas da marca '{marca_real}'"

# Testar
print(contar_armas_marca.invoke({"marca": "TAURUS"}))

Encontrei 17760 armas da marca 'TAURUS ARMAS S.A.                                                     '


In [5]:
# TOOL 2: Contar armas por calibre
@tool
def contar_armas_calibre(calibre: str) -> str:
    """
    Conta quantas armas de um calibre específico existem.
    
    Args:
        calibre: Calibre da arma (ex: 9mm, .38, .40)
    """
    df = carregar_csv()
    resultado = df[df['CALIBRE_ARMA'].str.contains(calibre, case=False, na=False)]
    total = len(resultado)
    
    if total == 0:
        return f"Não encontrei armas calibre '{calibre}'"
    
    return f"Encontrei {total} armas calibre '{calibre}'"

# Testar
print(contar_armas_calibre.invoke({"calibre": "9mm"}))

Encontrei 275 armas calibre '9mm'


In [6]:
# TOOL 3: Contar por tipo de ocorrência
@tool
def contar_armas_tipo(tipo: str) -> str:
    """
    Conta ocorrências por tipo (FURTO, ROUBO, APREENSAO).
    
    Args:
        tipo: Tipo de ocorrência
    """
    df = carregar_csv()
    resultado = df[df['TIPO_OCORRENCIA'].str.contains(tipo, case=False, na=False)]
    total = len(resultado)
    
    if total == 0:
        return f"Não encontrei ocorrências tipo '{tipo}'"
    
    tipo_real = resultado['TIPO_OCORRENCIA'].iloc[0]
    return f"Encontrei {total} ocorrências tipo '{tipo_real}'"

# Testar
print(contar_armas_tipo.invoke({"tipo": "ROUBO"}))

Encontrei 80 ocorrências tipo 'Roubo de Arma de Fogo                                                     '


In [7]:
# TOOL 4: Contar combinado (marca + tipo)
@tool
def contar_armas_combinado(marca: str, tipo: str) -> str:
    """
    Conta armas filtrando por marca E tipo de ocorrência.
    
    Args:
        marca: Nome da marca
        tipo: Tipo de ocorrência
    """
    df = carregar_csv()
    
    resultado = df[
        df['MARCA_ARMA'].str.contains(marca, case=False, na=False) &
        df['TIPO_OCORRENCIA'].str.contains(tipo, case=False, na=False)
    ]
    
    total = len(resultado)
    
    if total == 0:
        return f"Não encontrei armas '{marca}' tipo '{tipo}'"
    
    marca_real = resultado['MARCA_ARMA'].iloc[0]
    tipo_real = resultado['TIPO_OCORRENCIA'].iloc[0]
    return f"Encontrei {total} armas '{marca_real}' tipo '{tipo_real}'"

# Testar
print(contar_armas_combinado.invoke({"marca": "GLOCK", "tipo": "ROUBO"}))

Encontrei 5 armas 'GLOCK GMBH (ÁUSTRIA)                                                  ' tipo 'Roubo de Arma de Fogo                                                     '


In [8]:
# TOOL 5: Ranking de marcas
@tool
def ranking_marcas() -> str:
    """
    Retorna TOP 5 marcas mais registradas.
    """
    df = carregar_csv()
    ranking = df['MARCA_ARMA'].value_counts().head(5)
    
    resultado = "TOP 5 MARCAS MAIS REGISTRADAS:\n"
    for i, (marca, total) in enumerate(ranking.items(), 1):
        resultado += f"  {i}º - {marca}: {total} armas\n"
    
    return resultado

# Testar
print(ranking_marcas.invoke({}))

TOP 5 MARCAS MAIS REGISTRADAS:
  1º - TAURUS ARMAS S.A.                                                     : 17748 armas
  2º - ROSSI (AMADEO ROSSI S.A.)                                             : 16646 armas
  3º - CBC (COMPANHIA BRASILEIRA DE CARTUCHOS)                               : 9853 armas
  4º - BOITO (E.R. AMANTINO & CIA)                                           : 8200 armas
  5º - SMITH & WESSON                                                        : 2329 armas



In [9]:
# TOOL 6: Ranking de calibres
@tool
def ranking_calibres() -> str:
    """
    Retorna TOP 5 calibres mais comuns.
    """
    df = carregar_csv()
    ranking = df['CALIBRE_ARMA'].value_counts().head(5)
    
    resultado = "TOP 5 CALIBRES MAIS COMUNS:\n"
    for i, (calibre, total) in enumerate(ranking.items(), 1):
        resultado += f"  {i}º - {calibre}: {total} armas\n"
    
    return resultado

# Testar
print(ranking_calibres.invoke({}))

TOP 5 CALIBRES MAIS COMUNS:
  1º - .32                                               : 13814 armas
  2º - .38                                               : 11761 armas
  3º - .22 LR                                            : 10957 armas
  4º - 28                                                : 6237 armas
  5º - 36                                                : 5374 armas



In [10]:
# TOOL 7: Estatísticas gerais
@tool
def estatisticas_gerais() -> str:
    """
    Retorna estatísticas gerais do banco SINARM.
    """
    df = carregar_csv()
    
    total_registros = len(df)
    total_marcas = df['MARCA_ARMA'].nunique()
    total_calibres = df['CALIBRE_ARMA'].nunique()
    marca_mais_comum = df['MARCA_ARMA'].value_counts().index[0]
    calibre_mais_comum = df['CALIBRE_ARMA'].value_counts().index[0]
    tipo_mais_comum = df['TIPO_OCORRENCIA'].value_counts().index[0]
    
    resultado = "ESTATISTICAS GERAIS DO SINARM:\n"
    resultado += "\nTOTAIS:\n"
    resultado += f"  - Registros: {total_registros}\n"
    resultado += f"  - Marcas diferentes: {total_marcas}\n"
    resultado += f"  - Calibres diferentes: {total_calibres}\n"
    resultado += "\nMAIS COMUNS:\n"
    resultado += f"  - Marca: {marca_mais_comum}\n"
    resultado += f"  - Calibre: {calibre_mais_comum}\n"
    resultado += f"  - Tipo: {tipo_mais_comum}\n"
    
    return resultado

# Testar
print(estatisticas_gerais.invoke({}))

ESTATISTICAS GERAIS DO SINARM:

TOTAIS:
  - Registros: 74758
  - Marcas diferentes: 1177
  - Calibres diferentes: 93

MAIS COMUNS:
  - Marca: TAURUS ARMAS S.A.                                                     
  - Calibre: .32                                               
  - Tipo:                                                                           



In [11]:
# TOOL 8: Distribuição marca por tipo
@tool
def distribuicao_marca_por_tipo(marca: str) -> str:
    """
    Mostra distribuição de uma marca por tipo de ocorrência.
    
    Args:
        marca: Nome da marca
    """
    df = carregar_csv()
    
    resultado_marca = df[df['MARCA_ARMA'].str.contains(marca, case=False, na=False)]
    
    if len(resultado_marca) == 0:
        return f"Não encontrei armas da marca '{marca}'"
    
    marca_real = resultado_marca['MARCA_ARMA'].iloc[0]
    distribuicao = resultado_marca['TIPO_OCORRENCIA'].value_counts()
    total = len(resultado_marca)
    
    resultado = f"DISTRIBUICAO DE {marca_real} POR TIPO:\n"
    resultado += f"  Total: {total} armas\n\n"
    
    for tipo, qtd in distribuicao.items():
        percentual = (qtd / total) * 100
        resultado += f"  - {tipo}: {qtd} armas ({percentual:.1f}%)\n"
    
    return resultado

# Testar
print(distribuicao_marca_por_tipo.invoke({"marca": "BERETTA"}))

DISTRIBUICAO DE BERETTA (PIETRO BERETTA S.P.A.)                                        POR TIPO:
  Total: 1823 armas

  -                                                                           : 1798 armas (98.6%)
  - Campanha do Desarmamento                                                  : 11 armas (0.6%)
  - Extravio/Perda de Arma de Fogo                                            : 3 armas (0.2%)
  - Remetida ao Exército para destruição                                      : 3 armas (0.2%)
  - Apreensão de Arma de Fogo                                                 : 2 armas (0.1%)
  - Arrecadação                                                               : 2 armas (0.1%)
  - Apostilada no Exercito                                                    : 1 armas (0.1%)
  - Roubo de Arma de Fogo                                                     : 1 armas (0.1%)
  - Furto de Arma de Fogo                                                     : 1 armas (0.1%)
  - Recuperação de Arm

---

## ✅ CHECKPOINT 1

**Validação:**
- [ ] 8 tools do E3 funcionando?
- [ ] Dados CSV carregados?
- [ ] Testes passaram?

**Se tudo OK, prossiga para PARTE 2: RAG**

---

## PARTE 2: CONCEITOS RAG

### O que é RAG?

**RAG = Retrieval-Augmented Generation**

Combina duas etapas:

1. **RETRIEVAL (Recuperação)**
   - Buscar documentos relevantes
   - Usar embeddings + similaridade
   - Retornar TOP-K mais similares

2. **GENERATION (Geração)**
   - LLM gera resposta
   - Baseado nos documentos recuperados
   - Resposta fundamentada

### Por que usar RAG?

**Vantagens:**
- ✅ Respostas baseadas em documentos
- ✅ Sem necessidade de fine-tuning
- ✅ Atualização fácil (adicionar documentos)
- ✅ Transparência (mostra fonte)

**Desvantagens:**
- ❌ Depende da qualidade dos documentos
- ❌ Busca pode falhar
- ❌ Mais lento que consulta direta

### RAG vs Fine-tuning

| Aspecto | RAG | Fine-tuning |
|---------|-----|-------------|
| **Custo** | Baixo | Alto |
| **Tempo** | Rápido | Lento |
| **Atualização** | Fácil | Difícil |
| **Transparência** | Alta | Baixa |
| **Precisão** | Boa | Excelente |

### Quando usar RAG?

✅ **Use RAG quando:**
- Documentos mudam frequentemente
- Precisa citar fontes
- Orçamento limitado
- Resposta deve ser factual

❌ **Use Fine-tuning quando:**
- Precisa de estilo específico
- Dados são estáticos
- Orçamento disponível
- Latência crítica

---

## PASSO 4: Preparar Documentos Conceituais

Vamos carregar documentos que explicam conceitos.

In [16]:
# Listar documentos disponíveis
caminho_docs = "../01_DADOS/documentos_conceituais/"

if os.path.exists(caminho_docs):
    arquivos = [f for f in os.listdir(caminho_docs) if f.endswith('.txt')]
    print(f"📚 Documentos encontrados: {len(arquivos)}")
    for arquivo in arquivos:
        print(f"  - {arquivo}")
else:
    print(f"⚠️ Pasta não encontrada: {caminho_docs}")
    print("Criando documentos de exemplo...")
    
    # Criar pasta se não existir
    os.makedirs(caminho_docs, exist_ok=True)
    
    # Criar documentos de exemplo
    docs_exemplo = {
        "conceito_calibre.txt": """Calibre é a medida do diâmetro interno do cano de uma arma de fogo.
        É expresso em milímetros (mm) ou polegadas.
        Exemplos comuns: 9mm, .38, .40, .45, .380.
        O calibre determina o tamanho do projétil que a arma pode disparar.""",
        
        "conceito_arma_fogo.txt": """Arma de fogo é um dispositivo que utiliza a energia de gases em expansão
        para lançar um projétil a alta velocidade.
        Tipos principais: pistolas, revólveres, rifles, espingardas.
        Componentes: cano, gatilho, tambor/carregador, mecanismo de disparo.""",
        
        "conceito_sinarm.txt": """SINARM é o Sistema Nacional de Armas.
        Gerenciado pela Polícia Federal do Brasil.
        Registra e controla armas de fogo em território nacional.
        Objetivo: rastrear armas, prevenir crimes, auxiliar investigações."""
    }
    
    for nome, conteudo in docs_exemplo.items():
        with open(os.path.join(caminho_docs, nome), 'w', encoding='utf-8') as f:
            f.write(conteudo)
    
    print(f"✅ {len(docs_exemplo)} documentos criados!")

📚 Documentos encontrados: 5
  - calibres_armas.txt
  - marcas_armas.txt
  - sistema_sinarm.txt
  - tipos_armas.txt
  - rag_conceitos.txt


In [17]:
# Ler todos os documentos
def carregar_documentos():
    """
    Carrega todos os documentos conceituais.
    
    Returns:
        dict: {nome_arquivo: conteudo}
    """
    documentos = {}
    
    for arquivo in os.listdir(caminho_docs):
        if arquivo.endswith('.txt'):
            caminho_completo = os.path.join(caminho_docs, arquivo)
            with open(caminho_completo, 'r', encoding='utf-8') as f:
                documentos[arquivo] = f.read()
    
    return documentos

# Carregar
documentos = carregar_documentos()

print(f"📚 {len(documentos)} documentos carregados:\n")
for nome, conteudo in documentos.items():
    print(f"📄 {nome}")
    print(f"   Tamanho: {len(conteudo)} caracteres")
    print(f"   Preview: {conteudo[:100]}...\n")

📚 5 documentos carregados:

📄 calibres_armas.txt
   Tamanho: 5338 caracteres
   Preview: # Calibres de Armas de Fogo

## O que é Calibre?

Calibre é a medida do diâmetro interno do cano de ...

📄 marcas_armas.txt
   Tamanho: 8120 caracteres
   Preview: # Marcas de Armas de Fogo

## Principais Marcas Brasileiras

### Taurus (Forjas Taurus S.A.)

**Hist...

📄 sistema_sinarm.txt
   Tamanho: 8376 caracteres
   Preview: # Sistema SINARM - Sistema Nacional de Armas

## O que é o SINARM?

O SINARM (Sistema Nacional de Ar...

📄 tipos_armas.txt
   Tamanho: 8934 caracteres
   Preview: # Tipos de Armas de Fogo

## Classificação Geral

As armas de fogo são classificadas de acordo com d...

📄 rag_conceitos.txt
   Tamanho: 10816 caracteres
   Preview: # RAG - Retrieval-Augmented Generation

## O que é RAG?

RAG (Retrieval-Augmented Generation) é uma ...



---

## PASSO 5: Gerar Embeddings (TF-IDF)

**O que são embeddings?**
- Representação numérica de texto
- Vetores que capturam significado
- Textos similares = vetores próximos

**TF-IDF vs Transformers:**
- **TF-IDF:** Rápido, leve, baseado em frequência
- **Transformers:** Lento, pesado, baseado em contexto

**Para E4, usaremos TF-IDF** (mais simples e rápido)

In [19]:
# Preparar textos para vetorização
nomes_docs = list(documentos.keys())
textos_docs = list(documentos.values())

print(f"📊 Preparando {len(textos_docs)} documentos para vetorização...")

# Criar vetorizador TF-IDF
vectorizer = TfidfVectorizer(
    max_features=100,  # Máximo de features
    stop_words=None,   # Sem stop words (português não tem lista padrão)
    ngram_range=(1, 2) # Unigramas e bigramas
)

# Gerar embeddings
embeddings_docs = vectorizer.fit_transform(textos_docs)

print(f"✅ Embeddings gerados!")
print(f"   Forma da matriz: {embeddings_docs.shape}")
print(f"   ({embeddings_docs.shape[0]} documentos x {embeddings_docs.shape[1]} features)")

📊 Preparando 5 documentos para vetorização...
✅ Embeddings gerados!
   Forma da matriz: (5, 100)
   (5 documentos x 100 features)


In [20]:
# Visualizar features mais importantes
feature_names = vectorizer.get_feature_names_out()

print(f"\n🔍 Top 10 features (palavras importantes):\n")
for i, doc_name in enumerate(nomes_docs):
    print(f"📄 {doc_name}:")
    
    # Pegar scores TF-IDF do documento
    doc_vector = embeddings_docs[i].toarray()[0]
    
    # Top 5 features
    top_indices = doc_vector.argsort()[-5:][::-1]
    top_features = [(feature_names[idx], doc_vector[idx]) for idx in top_indices if doc_vector[idx] > 0]
    
    for feature, score in top_features:
        print(f"   - {feature}: {score:.3f}")
    print()


🔍 Top 10 features (palavras importantes):

📄 calibres_armas.txt:
   - de: 0.334
   - efetivo: 0.294
   - alcance efetivo: 0.294
   - metros: 0.273
   - alcance: 0.237

📄 marcas_armas.txt:
   - de: 0.371
   - em: 0.281
   - eua: 0.273
   - história: 0.252
   - 9mm: 0.237

📄 sistema_sinarm.txt:
   - de: 0.718
   - sinarm: 0.327
   - arma: 0.261
   - registro: 0.249
   - armas: 0.249

📄 tipos_armas.txt:
   - de: 0.477
   - ação: 0.334
   - uso: 0.239
   - exemplo: 0.238
   - gatilho: 0.230

📄 rag_conceitos.txt:
   - documentos: 0.408
   - pergunta: 0.404
   - de: 0.366
   - rag: 0.344
   - embeddings: 0.263



---

## PASSO 6: Testar Similaridade

Vamos testar se conseguimos encontrar documentos similares a uma pergunta.

In [22]:
def buscar_documentos_similares(pergunta: str, top_k: int = 2):
    """
    Busca documentos mais similares à pergunta.
    
    Args:
        pergunta: Pergunta do usuário
        top_k: Quantos documentos retornar
    
    Returns:
        list: [(nome_doc, conteudo, score)]
    """
    # Vetorizar pergunta
    pergunta_vector = vectorizer.transform([pergunta])
    
    # Calcular similaridade com todos os documentos
    similaridades = cosine_similarity(pergunta_vector, embeddings_docs)[0]
    
    # Pegar top-k mais similares
    top_indices = similaridades.argsort()[-top_k:][::-1]
    
    resultados = []
    for idx in top_indices:
        nome = nomes_docs[idx]
        conteudo = textos_docs[idx]
        score = similaridades[idx]
        resultados.append((nome, conteudo, score))
    
    return resultados

# Testar com perguntas
perguntas_teste = [
    "O que é calibre?",
    "Como funciona o SINARM?",
    "O que é uma arma de fogo?"
]

for pergunta in perguntas_teste:
    print(f"\n❓ Pergunta: {pergunta}")
    print(f"{'='*60}")
    
    resultados = buscar_documentos_similares(pergunta, top_k=2)
    
    for i, (nome, conteudo, score) in enumerate(resultados, 1):
        print(f"\n{i}. 📄 {nome} (similaridade: {score:.3f})")
        print(f"   {conteudo[:150]}...")


❓ Pergunta: O que é calibre?

1. 📄 calibres_armas.txt (similaridade: 0.221)
   # Calibres de Armas de Fogo

## O que é Calibre?

Calibre é a medida do diâmetro interno do cano de uma arma de fogo, geralmente expressa em milímetro...

2. 📄 rag_conceitos.txt (similaridade: 0.095)
   # RAG - Retrieval-Augmented Generation

## O que é RAG?

RAG (Retrieval-Augmented Generation) é uma técnica que combina busca de informações (retrieva...

❓ Pergunta: Como funciona o SINARM?

1. 📄 sistema_sinarm.txt (similaridade: 0.327)
   # Sistema SINARM - Sistema Nacional de Armas

## O que é o SINARM?

O SINARM (Sistema Nacional de Armas) é um banco de dados nacional criado pela Lei ...

2. 📄 rag_conceitos.txt (similaridade: 0.033)
   # RAG - Retrieval-Augmented Generation

## O que é RAG?

RAG (Retrieval-Augmented Generation) é uma técnica que combina busca de informações (retrieva...

❓ Pergunta: O que é uma arma de fogo?

1. 📄 sistema_sinarm.txt (similaridade: 0.567)
   # Sistema SINARM - Sistema Nac

---

## ✅ CHECKPOINT 2

**Validação:**
- [ ] Documentos carregados?
- [ ] Embeddings gerados?
- [ ] Busca por similaridade funciona?
- [ ] Documentos corretos são retornados?

**Se tudo OK, prossiga para PARTE 3: TOOL RAG**

---

## PARTE 3: CRIAR TOOL RAG

Agora vamos criar a **9ª tool** que usa RAG para responder perguntas conceituais.

In [23]:
# TOOL 9: Buscar conceito (RAG)
@tool
def buscar_conceito(pergunta: str) -> str:
    """
    Busca resposta para perguntas conceituais usando RAG.
    
    Args:
        pergunta: Pergunta conceitual (ex: "O que é calibre?")
    
    Returns:
        Resposta baseada nos documentos mais relevantes
    """
    # Buscar documentos similares
    resultados = buscar_documentos_similares(pergunta, top_k=2)
    
    if not resultados or resultados[0][2] < 0.1:  # Score muito baixo
        return "Desculpe, não encontrei informações sobre isso nos documentos."
    
    # Montar resposta baseada nos documentos
    resposta = f"📚 Baseado nos documentos encontrados:\n\n"
    
    for i, (nome, conteudo, score) in enumerate(resultados, 1):
        resposta += f"{i}. {conteudo}\n\n"
    
    resposta += f"\n📄 Fontes: {', '.join([r[0] for r in resultados])}"
    
    return resposta

# Testar tool RAG
print("🧪 Testando tool RAG:\n")
print(buscar_conceito.invoke({"pergunta": "O que é calibre?"}))

🧪 Testando tool RAG:

📚 Baseado nos documentos encontrados:

1. # Calibres de Armas de Fogo

## O que é Calibre?

Calibre é a medida do diâmetro interno do cano de uma arma de fogo, geralmente expressa em milímetros (mm) ou polegadas. O calibre determina o tamanho do projétil que a arma pode disparar.

## Principais Calibres no Brasil

### Calibres Comuns em Pistolas

**9mm (9x19mm Parabellum)**
- Calibre mais popular no mundo
- Usado por forças policiais e militares
- Boa capacidade de munição (15-17 cartuchos)
- Recuo moderado, fácil controle
- Alcance efetivo: 50 metros

**.40 S&W (Smith & Wesson)**
- Desenvolvido para polícia americana
- Maior poder de parada que 9mm
- Recuo mais forte
- Capacidade: 12-15 cartuchos
- Alcance efetivo: 50 metros

**.45 ACP (Automatic Colt Pistol)**
- Calibre tradicional americano
- Grande poder de parada
- Recuo significativo
- Capacidade: 7-10 cartuchos
- Alcance efetivo: 50 metros

**.380 ACP (9mm Curto)**
- Calibre compacto
- Usado em armas de por

---

## PASSO 7: Roteador Expandido (E3 + E4)

Agora precisamos de um roteador que decide:
- **Pergunta estruturada** → Usar tools E3 (1-8)
- **Pergunta conceitual** → Usar tool RAG (9)

In [24]:
def classificar_pergunta(pergunta: str) -> str:
    """
    Classifica se a pergunta é estruturada ou conceitual.
    
    Args:
        pergunta: Pergunta do usuário
    
    Returns:
        'estruturada' ou 'conceitual'
    """
    pergunta_lower = pergunta.lower()
    
    # Palavras-chave para perguntas conceituais
    palavras_conceituais = [
        'o que é', 'o que são', 'como funciona', 'explique',
        'defina', 'definição', 'conceito', 'significado',
        'diferença entre', 'qual a diferença'
    ]
    
    # Palavras-chave para perguntas estruturadas
    palavras_estruturadas = [
        'quantas', 'quanto', 'total', 'top', 'ranking',
        'estatística', 'distribuição', 'marca', 'calibre',
        'taurus', 'glock', 'beretta', '9mm', '.38', '.40'
    ]
    
    # Verificar conceitual
    for palavra in palavras_conceituais:
        if palavra in pergunta_lower:
            return 'conceitual'
    
    # Verificar estruturada
    for palavra in palavras_estruturadas:
        if palavra in pergunta_lower:
            return 'estruturada'
    
    # Padrão: conceitual (se não identificou)
    return 'conceitual'

# Testar classificador
perguntas_teste = [
    "Quantas armas Taurus?",
    "O que é calibre?",
    "Top 5 marcas",
    "Como funciona o SINARM?",
    "Glock roubadas"
]

print("🧪 Testando classificador:\n")
for pergunta in perguntas_teste:
    tipo = classificar_pergunta(pergunta)
    print(f"❓ {pergunta}")
    print(f"   → {tipo}\n")

🧪 Testando classificador:

❓ Quantas armas Taurus?
   → estruturada

❓ O que é calibre?
   → conceitual

❓ Top 5 marcas
   → estruturada

❓ Como funciona o SINARM?
   → conceitual

❓ Glock roubadas
   → estruturada



---

## PASSO 8: Agente Completo E3+E4

Vamos criar uma função que integra tudo.

In [25]:
def processar_pergunta(pergunta: str) -> str:
    """
    Processa pergunta usando agente E3+E4.
    
    Args:
        pergunta: Pergunta do usuário
    
    Returns:
        Resposta do agente
    """
    # Validação básica
    if len(pergunta) < 3:
        return "❌ Pergunta muito curta. Digite pelo menos 3 caracteres."
    
    if len(pergunta) > 500:
        return "❌ Pergunta muito longa. Máximo 500 caracteres."
    
    # Classificar pergunta
    tipo = classificar_pergunta(pergunta)
    
    print(f"🔍 Tipo detectado: {tipo}")
    
    # Rotear para tool apropriada
    if tipo == 'conceitual':
        # Usar RAG
        return buscar_conceito.invoke({"pergunta": pergunta})
    
    else:
        # Usar tools E3 (roteador do E3)
        pergunta_lower = pergunta.lower()
        
        # Prioridade 1: Estatísticas gerais
        if any(palavra in pergunta_lower for palavra in ['resumo', 'estatística', 'geral', 'total']):
            return estatisticas_gerais.invoke({})
        
        # Prioridade 2: Distribuição
        if 'distribuição' in pergunta_lower or 'distribuicao' in pergunta_lower:
            # Extrair marca
            marcas = ['taurus', 'glock', 'beretta', 'imbel']
            for marca in marcas:
                if marca in pergunta_lower:
                    return distribuicao_marca_por_tipo.invoke({"marca": marca})
        
        # Prioridade 3: Rankings
        if 'top' in pergunta_lower or 'ranking' in pergunta_lower:
            if 'marca' in pergunta_lower:
                return ranking_marcas.invoke({})
            elif 'calibre' in pergunta_lower:
                return ranking_calibres.invoke({})
        
        # Prioridade 4: Consultas combinadas
        marcas = ['taurus', 'glock', 'beretta', 'imbel']
        tipos = ['furto', 'roubo', 'apreensao']
        
        marca_encontrada = None
        tipo_encontrado = None
        
        for marca in marcas:
            if marca in pergunta_lower:
                marca_encontrada = marca
                break
        
        for tipo in tipos:
            if tipo in pergunta_lower:
                tipo_encontrado = tipo
                break
        
        if marca_encontrada and tipo_encontrado:
            return contar_armas_combinado.invoke({"marca": marca_encontrada, "tipo": tipo_encontrado})
        
        # Prioridade 5: Consultas simples
        if marca_encontrada:
            return contar_armas_marca.invoke({"marca": marca_encontrada})
        
        calibres = ['9mm', '.38', '.40', '.45', '.380']
        for calibre in calibres:
            if calibre in pergunta_lower:
                return contar_armas_calibre.invoke({"calibre": calibre})
        
        if tipo_encontrado:
            return contar_armas_tipo.invoke({"tipo": tipo_encontrado})
        
        return "❌ Não entendi a pergunta. Tente reformular."

print("✅ Agente E3+E4 criado!")

✅ Agente E3+E4 criado!


---

## PASSO 9: Testes Completos

Vamos testar o agente com perguntas estruturadas E conceituais.

In [26]:
# Testes estruturados (E3)
print("="*60)
print("TESTES ESTRUTURADOS (E3)")
print("="*60)

perguntas_estruturadas = [
    "Quantas armas Taurus?",
    "Top 5 marcas",
    "Glock roubadas",
    "Distribuição Beretta"
]

for pergunta in perguntas_estruturadas:
    print(f"\n❓ {pergunta}")
    print(f"{'-'*60}")
    resposta = processar_pergunta(pergunta)
    print(resposta)

TESTES ESTRUTURADOS (E3)

❓ Quantas armas Taurus?
------------------------------------------------------------
🔍 Tipo detectado: estruturada
Encontrei 17760 armas da marca 'TAURUS ARMAS S.A.                                                     '

❓ Top 5 marcas
------------------------------------------------------------
🔍 Tipo detectado: estruturada
TOP 5 MARCAS MAIS REGISTRADAS:
  1º - TAURUS ARMAS S.A.                                                     : 17748 armas
  2º - ROSSI (AMADEO ROSSI S.A.)                                             : 16646 armas
  3º - CBC (COMPANHIA BRASILEIRA DE CARTUCHOS)                               : 9853 armas
  4º - BOITO (E.R. AMANTINO & CIA)                                           : 8200 armas
  5º - SMITH & WESSON                                                        : 2329 armas


❓ Glock roubadas
------------------------------------------------------------
🔍 Tipo detectado: estruturada
Encontrei 726 armas da marca 'GLOCK GMBH (ÁUSTRIA)     

In [27]:
# Testes conceituais (E4 - RAG)
print("\n" + "="*60)
print("TESTES CONCEITUAIS (E4 - RAG)")
print("="*60)

perguntas_conceituais = [
    "O que é calibre?",
    "Como funciona o SINARM?",
    "O que é uma arma de fogo?"
]

for pergunta in perguntas_conceituais:
    print(f"\n❓ {pergunta}")
    print(f"{'-'*60}")
    resposta = processar_pergunta(pergunta)
    print(resposta)


TESTES CONCEITUAIS (E4 - RAG)

❓ O que é calibre?
------------------------------------------------------------
🔍 Tipo detectado: conceitual
📚 Baseado nos documentos encontrados:

1. # Calibres de Armas de Fogo

## O que é Calibre?

Calibre é a medida do diâmetro interno do cano de uma arma de fogo, geralmente expressa em milímetros (mm) ou polegadas. O calibre determina o tamanho do projétil que a arma pode disparar.

## Principais Calibres no Brasil

### Calibres Comuns em Pistolas

**9mm (9x19mm Parabellum)**
- Calibre mais popular no mundo
- Usado por forças policiais e militares
- Boa capacidade de munição (15-17 cartuchos)
- Recuo moderado, fácil controle
- Alcance efetivo: 50 metros

**.40 S&W (Smith & Wesson)**
- Desenvolvido para polícia americana
- Maior poder de parada que 9mm
- Recuo mais forte
- Capacidade: 12-15 cartuchos
- Alcance efetivo: 50 metros

**.45 ACP (Automatic Colt Pistol)**
- Calibre tradicional americano
- Grande poder de parada
- Recuo significativo
- Capac

---

## PASSO 10: Modo Interativo

Teste o agente de forma interativa!

In [28]:
def modo_interativo():
    """
    Modo interativo para testar o agente.
    """
    print("\n" + "="*60)
    print("AGENTE SINARM E3+E4 - MODO INTERATIVO")
    print("="*60)
    print("\nDigite suas perguntas (ou 'sair' para encerrar)\n")
    
    contador = 0
    
    while True:
        pergunta = input("\n❓ Sua pergunta: ").strip()
        
        if pergunta.lower() in ['sair', 'exit', 'quit']:
            print("\n👋 Até logo!")
            break
        
        if not pergunta:
            continue
        
        contador += 1
        print(f"\n{'='*60}")
        print(f"PERGUNTA #{contador}")
        print(f"{'='*60}")
        
        resposta = processar_pergunta(pergunta)
        print(f"\n{resposta}")

# Descomentar para usar modo interativo
# modo_interativo()

---

## ✅ CHECKPOINT FINAL

**Validação:**
- [ ] Agente responde perguntas estruturadas (E3)?
- [ ] Agente responde perguntas conceituais (E4 - RAG)?
- [ ] Classificador funciona corretamente?
- [ ] Modo interativo funcional?

**Se tudo OK, parabéns! Você completou o E4!** 🎉

---

## CONCLUSÃO E PRÓXIMOS PASSOS

### O que construímos no E4:

**Mantido do E3:**
- ✅ 8 tools para dados estruturados
- ✅ Cache com @lru_cache
- ✅ Validação de segurança

**Adicionado no E4:**
- ✅ Tool RAG para perguntas conceituais
- ✅ Embeddings com TF-IDF
- ✅ Busca por similaridade
- ✅ Classificador de perguntas
- ✅ Integração E3+E4

### Comparação E3 vs E4:

| Aspecto | E3 | E4 |
|---------|----|----||
| **Tools** | 8 | 9 |
| **Dados** | Estruturados | Estruturados + Não-estruturados |
| **Busca** | Filtros pandas | Filtros + Busca semântica |
| **Perguntas** | Estruturadas | Estruturadas + Conceituais |

### Próximos Passos (E5):

1. **Memory** - Memória conversacional
2. **Multi-Agent** - Múltiplos agentes especializados
3. **Coordenação** - Agentes trabalhando juntos

### Para Consolidar:

1. Extrair código deste notebook
2. Criar `agente_v4_7_completo.py`
3. Adicionar 3 modos de execução
4. Testar em produção

---

**Parabéns por completar o E4!** 🎉🚀